### **Title:** **Luong Attention (Multiplicative Attention)**

### **Objectives:**

* To implement an RNN Encoder-Decoder architecture with **Luong (Multiplicative) Attention** for English-French machine translation using PyTorch.
* To understand how the **Luong Attention** mechanism improves translation quality by enabling the decoder to focus on the most relevant encoder hidden states during each decoding step using multiplicative alignment scores.

### **Theory:**

This project develops an RNN-based Encoder–Decoder model with **Luong (Multiplicative) Attention** for English-French machine translation using PyTorch. The encoder first converts each word in the input French sentence into a vector embedding and processes the sequence through an RNN to generate a sequence of hidden states representing the input sentence. Unlike the basic Encoder–Decoder model, which relies only on the encoder's final hidden state, the **Luong Attention** mechanism computes attention after the decoder RNN generates its hidden state. At each decoding step, the decoder hidden state is compared with all encoder hidden states using a **multiplicative (dot-product or general) alignment function** to calculate attention scores. These scores are normalized using the **Softmax** function to produce attention weights, which are then used to compute a context vector as a weighted sum of the encoder hidden states. The context vector is concatenated with the decoder output to generate the final prediction, allowing the decoder to focus on the most relevant parts of the input sentence during translation. This approach improves translation performance, especially for longer and more complex sentences, while being computationally more efficient than additive attention. During training, **teacher forcing** is employed, where the actual target word is provided as the next decoder input instead of the model's previous prediction, improving convergence speed and training stability. The model is optimized using the **Adam** optimizer, and the **Negative Log Likelihood (NLL)** loss function is used to measure prediction errors and update the network parameters through backpropagation.


In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
from torch.utils.data import TensorDataset

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [7]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [9]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am'] 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['j arrive chez moi', 'i m coming home']


15

In [10]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

In [23]:
class LuongDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(LuongDecoderRNN, self).__init__()

        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)

        self.dropout = nn.Dropout(dropout_p)

        self.rnn = nn.RNN(
            hidden_size,
            hidden_size,
            batch_first=True
        )

        self.attention = LuongDotAttention(hidden_size)

        self.concat = nn.Linear(
            hidden_size * 2,
            hidden_size
        )

        self.out = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self,
                encoder_outputs,
                encoder_hidden,
                target_tensor=None):

        batch_size = encoder_outputs.size(0)

        decoder_input = torch.full(
            (batch_size, 1),
            SOS_token,
            dtype=torch.long,
            device=device
        )

        decoder_hidden = encoder_hidden

        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):

            decoder_output, decoder_hidden, attn = self.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs
            )

            decoder_outputs.append(decoder_output)
            attentions.append(attn)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self,
                     input,
                     hidden,
                     encoder_outputs):

        embedded = self.dropout(
            self.embedding(input)
        )

        # Decoder RNN first
        rnn_output, hidden = self.rnn(
            embedded,
            hidden
        )

        # Prepare query
        query = rnn_output

        # Luong attention
        context, attn_weights = self.attention(
            query,
            encoder_outputs
        )

        # Concatenate context and decoder output
        concat_input = torch.cat(
            (rnn_output, context),
            dim=2
        )

        concat_output = torch.tanh(
            self.concat(concat_input)
        )

        output = self.out(concat_output)

        return output, hidden, attn_weights

In [24]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [25]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [26]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [27]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [28]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [29]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [30]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [31]:
from torch.utils.data import TensorDataset
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = LuongDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 9s (- 6m 14s) (5 2%) 1.8766
0m 16s (- 5m 18s) (10 5%) 1.1018
0m 23s (- 4m 55s) (15 7%) 0.7596
0m 31s (- 4m 40s) (20 10%) 0.5093
0m 38s (- 4m 27s) (25 12%) 0.3479
0m 46s (- 4m 23s) (30 15%) 0.2555
0m 54s (- 4m 18s) (35 17%) 0.2077
1m 27s (- 5m 50s) (40 20%) 0.1764
1m 34s (- 5m 24s) (45 22%) 0.1584
1m 41s (- 5m 3s) (50 25%) 0.1465
1m 48s (- 4m 44s) (55 27%) 0.1375
1m 55s (- 4m 28s) (60 30%) 0.1302
2m 4s (- 4m 19s) (65 32%) 0.1194
2m 12s (- 4m 6s) (70 35%) 0.1158
2m 20s (- 3m 54s) (75 37%) 0.1100
2m 28s (- 3m 42s) (80 40%) 0.1067
2m 38s (- 3m 34s) (85 42%) 0.1028
2m 56s (- 3m 35s) (90 45%) 0.1006
3m 3s (- 3m 22s) (95 47%) 0.1024
3m 9s (- 3m 9s) (100 50%) 0.0997
3m 16s (- 2m 57s) (105 52%) 0.0955
3m 22s (- 2m 45s) (110 55%) 0.0939
3m 28s (- 2m 34s) (115 57%) 0.0923
3m 35s (- 2m 23s) (120 60%) 0.0895
3m 42s (- 2m 13s) (125 62%) 0.0862
3m 50s (- 2m 4s) (130 65%) 0.0

In [32]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je me sens stresse
= i m feeling stressed
< i m feeling stressed <EOS>

> je suis bourree
= i m sloshed
< i m sloshed <EOS>

> nous sommes prudentes
= we re careful
< we re happy <EOS>

> tu es tres attirante
= you re very attractive
< you re very attractive <EOS>

> nous sommes touches
= we re touched
< we re touched <EOS>

> nous sommes maries
= we re married
< we re married <EOS>

> vous etes celle la
= you are the one
< you are the one <EOS>

> vous etes fort contraries
= you re very upset
< you re very upset <EOS>

> je suis tres petit
= i m very short
< i m very short <EOS>

> je suis epuise
= i m exhausted
< i m full <EOS>



### **Discussion**

The implemented model demonstrates an RNN Encoder–Decoder with **Luong (Multiplicative) Attention** for English-French machine translation. The encoder converts the input sentence into hidden state representations, while the decoder generates the translation one word at a time. At each decoding step, Luong Attention computes attention scores between the decoder hidden state and all encoder hidden states to produce a context vector, enabling the decoder to focus on the most relevant parts of the input sentence. After preprocessing the dataset and training the model using **teacher forcing**, **Adam** optimizer, and **NLL loss**, the decreasing training loss indicates successful learning and improved translation performance.

### **Conclusion**

The RNN Encoder–Decoder model with **Luong Attention** improves machine translation by allowing the decoder to focus on relevant encoder hidden states during decoding. This reduces the limitations of a fixed-length context vector and produces more accurate translations. The implementation also provides a strong foundation for understanding modern attention-based neural machine translation models.
